# z011 - Variables con drift (PSI de z008): pruebas separadas

Cada versión cambia **una sola cosa** y se compara contra la Actual (el ajuste se hace antes de lags y deltas):

| Versión | Cambio | Motivo |
|---|---|---|
| Actual | nada | referencia |
| Sin fechas | se sacan `Master_fultimo_cierre` y `Visa_fultimo_cierre` | fechas de cierre: PSI ~7 entre 202104 y 202106 |
| Sueldo ajustado | `mpayroll` de 202106 × 2/3 | aguinaldo: en junio CONTINUA cobra 1,51× mayo; el PSI baja de 0,21 a 0,003 |
| Depósitos ajustados | `cpayroll_trx` de 202106 × (promedio de 202105 y 202107 / promedio de 202106) | aguinaldo como depósito aparte: 2,27 vs ~1,77 |
| Visa pagado suavizado | `Visa_mpagado` → promedio de los últimos 3 meses | patrón trimestral: % con pago > 0 es 25% en 202105 y 202108, ~9% el resto |

Chequeo previo (07/10): los BAJA+2 no reciben el aumento de junio (mediana 0,98× mayo, 34% cobra 0) → el ajuste
×2/3 es parejo para todos y no borra esa señal (la delta mayo→junio de los que se van queda más negativa).

Trial 34, 538 árboles, 10 semillas, train 202104, valid 202106.

In [1]:
# --- 1. Librerías y parámetros comunes ---
import numpy as np
import pandas as pd
import lightgbm as lgb

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO   = 27_500
MES_VALID = 202106
SEMILLAS  = [501419, 623353, 786349, 834569, 946331,   # las tuyas
             111119, 222227, 333331, 444443, 555557]  # 5 más (las mismas del ensamble)
N_PARTICIONES = 50        # Shuffle Split del test: particiones al azar del 50% de 202106

# Trial 34 de z005 (el mejor en promedio de 5 semillas)
params = {
    "objective": "binary", "learning_rate": 0.005, "min_data_in_leaf": 0,
    "feature_fraction": 0.5, "feature_fraction_bynode": 0.2, "max_bin": 31, "verbose": -1,
    "num_leaves": 137, "min_sum_hessian_in_leaf": 9.755574,
}
ARBOLES = 538

In [2]:
# --- 2. Cargar datos crudos (los lags se calculan después de ajustar) ---
crudo = pd.read_parquet("../competencia_01.parquet")
crudo["clase_binaria"] = (crudo["clase_ternaria"] == "BAJA+2").astype(int)
crudo = crudo.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)

no_usar = ["foto_mes", "clase_ternaria", "clase_binaria", "clase_binaria2"]
cols = [c for c in crudo.columns if c not in no_usar and c != "numero_de_cliente"]

# La ganancia se calcula con BAJA+2 de 202106 (el orden de filas es el mismo en todas las versiones)
valid_mask = (crudo["foto_mes"] == MES_VALID).values
ganancia_cliente = np.where(crudo.loc[valid_mask, "clase_ternaria"].values == "BAJA+2", GANANCIA_ACIERTO, -COSTO_ESTIMULO)

def con_lags(df):
    grupo = df.groupby("numero_de_cliente")[cols]
    nuevas = []
    for k in [1, 2]:
        lag = grupo.shift(k).astype("float32")
        delta = (df[cols] - lag).astype("float32")
        lag.columns   = [f"{c}_lag{k}" for c in cols]
        delta.columns = [f"{c}_delta{k}" for c in cols]
        nuevas += [lag, delta]
    return pd.concat([df] + nuevas, axis=1)

In [3]:
# --- 3. Funciones de evaluación ---
def metricas(prob, mascara=None):
    # Ganancia acumulada ordenando de mayor a menor probabilidad (opcional: solo una parte de los clientes)
    g = ganancia_cliente if mascara is None else ganancia_cliente[mascara]
    p = prob if mascara is None else prob[mascara]
    acum = np.cumsum(g[np.argsort(p)[::-1]])
    n = int(np.argmax(acum)) + 1
    return acum[n - 1] / 1e6, n, acum[min(11_999, len(acum) - 1)] / 1e6

def resumen(probs_por_version):
    """probs_por_version = {version: [prob_semilla1, ..., prob_semilla5]}"""
    # a) Por semilla: ganancia máxima, mejor N y top 12.000
    filas = []
    for v, lista in probs_por_version.items():
        for s, pr in zip(SEMILLAS, lista):
            g, n, g12 = metricas(pr)
            filas.append({"version": v, "semilla": s, "ganancia_max_M": round(g, 1), "mejor_n": n, "top12000_M": round(g12, 1)})
    por_semilla = pd.DataFrame(filas)
    tabla = por_semilla.groupby("version", sort=False)[["ganancia_max_M", "top12000_M", "mejor_n"]].agg(["mean", "std"]).round(1)

    # b) Shuffle Split del test: ensamble de las 5 semillas evaluado en 50 particiones del 50% de 202106
    rng = np.random.default_rng(SEMILLAS[0])
    ens = {v: np.mean(lista, axis=0) for v, lista in probs_por_version.items()}
    part = []
    for i in range(N_PARTICIONES):
        mascara = rng.random(len(ganancia_cliente)) < 0.5
        part.append({v: metricas(pr, mascara)[0] for v, pr in ens.items()})
    part = pd.DataFrame(part)
    tabla[("shuffle_split", "mean")] = part.mean().round(1)
    tabla[("shuffle_split", "std")] = part.std().round(1)
    tabla[("shuffle_split", "veces_mejor")] = part.idxmax(axis=1).value_counts().reindex(tabla.index).fillna(0).astype(int)
    tabla[("ensamble_5", "ganancia_max_M")] = [round(metricas(ens[v])[0], 1) for v in tabla.index]
    return por_semilla, tabla

def anotar(tabla, prefijo, meses_train, extra=""):
    ruta_res = "resultados_experimentos.csv"
    res = pd.read_csv(ruta_res, sep=";", decimal=",", encoding="utf-8-sig")
    res = res[~res["descripcion"].str.startswith(prefijo)]   # sin duplicar si se vuelve a correr
    for v, r in tabla.iterrows():
        nueva = {
            "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
            "descripcion": (f"{prefijo} {v}: prom. {len(SEMILLAS)} semillas; shuffle split {r[('shuffle_split', 'mean')]} "
                            f"(mejor en {r[('shuffle_split', 'veces_mejor')]}/{N_PARTICIONES}){extra}"),
            "meses_train": str(meses_train), "mes_valid": MES_VALID, "semilla": str(SEMILLAS),
            "params": str(params), "mejor_n": int(r[("mejor_n", "mean")]),
            "ganancia_max_M": r[("ganancia_max_M", "mean")], "ganancia_top12000_M": r[("top12000_M", "mean")],
        }
        res = pd.concat([res, pd.DataFrame([nueva])], ignore_index=True)
    res["n_enviados"] = res["n_enviados"].astype("Int64")
    res.to_csv(ruta_res, sep=";", decimal=",", index=False, encoding="utf-8-sig")
    print("Anotado en", ruta_res)

In [4]:
# --- 4. Versiones (una sola modificación cada una) y entrenamiento ---
fechas = ["Master_fultimo_cierre", "Visa_fultimo_cierre"]
es_junio = crudo["foto_mes"] == 202106

def sin_fechas(df):
    return df.drop(columns=fechas)

def sueldo_ajustado(df):
    df = df.copy()
    df["mpayroll"] = df["mpayroll"].astype("float64")
    df.loc[es_junio, "mpayroll"] = df.loc[es_junio, "mpayroll"] * 2 / 3
    return df

def depositos_ajustados(df):
    df = df.copy()
    prom = df[df["cpayroll_trx"] > 0].groupby("foto_mes")["cpayroll_trx"].mean()
    factor = (prom[202105] + prom[202107]) / 2 / prom[202106]
    print("Factor para cpayroll_trx de 202106:", round(factor, 3))
    df["cpayroll_trx"] = df["cpayroll_trx"].astype("float32")   # es entera: la pasamos a decimal para poder escalarla
    df.loc[es_junio, "cpayroll_trx"] = df.loc[es_junio, "cpayroll_trx"] * factor
    return df

def visa_suavizado(df):
    df = df.copy()
    df["Visa_mpagado"] = (df.groupby("numero_de_cliente")["Visa_mpagado"].rolling(3, min_periods=1).mean()
                            .reset_index(level=0, drop=True).sort_index().astype("float32"))
    return df

versiones = {"Actual": lambda df: df, "Sin fechas": sin_fechas, "Sueldo ajustado": sueldo_ajustado,
             "Depósitos ajustados": depositos_ajustados, "Visa pagado suavizado": visa_suavizado}
probs = {}
for version, f in versiones.items():
    datos_v = f(crudo)
    cols = [c for c in datos_v.columns if c not in no_usar and c != "numero_de_cliente"]   # con_lags usa cols
    df = con_lags(datos_v)
    variables = [c for c in df.columns if c not in no_usar]
    train = df[df["foto_mes"] == 202104]
    valid = df[valid_mask]
    datos = lgb.Dataset(train[variables], label=train["clase_binaria"])
    probs[version] = [lgb.train({**params, "seed": s}, datos, num_boost_round=ARBOLES).predict(valid[variables])
                      for s in SEMILLAS]
    del df, train, valid, datos, datos_v
    print(version, len(variables), "variables, listo")

Actual 761 variables, listo


Sin fechas 751 variables, listo


Sueldo ajustado 761 variables, listo


Factor para cpayroll_trx de 202106: 0.782


Depósitos ajustados 761 variables, listo


Visa pagado suavizado 761 variables, listo


In [5]:
# --- 5. Comparación: cada versión contra la Actual ---
por_semilla, tabla = resumen(probs)
g = por_semilla.pivot(index="semilla", columns="version", values="ganancia_max_M")
for v in g.columns.drop("Actual"):
    dif = g[v] - g["Actual"]
    print(f"{v}: diferencia promedio {dif.mean():+.1f} M, gana en {(dif > 0).sum()} de {len(dif)} semillas")
tabla

Depósitos ajustados: diferencia promedio +0.0 M, gana en 1 de 10 semillas
Sin fechas: diferencia promedio +1.6 M, gana en 6 de 10 semillas
Sueldo ajustado: diferencia promedio -0.6 M, gana en 0 de 10 semillas
Visa pagado suavizado: diferencia promedio -2.2 M, gana en 3 de 10 semillas


ganancia_max_M      top12000_M       mejor_n          \
                                mean  std       mean  std     mean     std   
version                                                                      
Actual                         399.3  2.6      391.7  3.7  13230.9  1224.2   
Sin fechas                     400.9  4.0      392.5  3.4  13172.8   940.6   
Sueldo ajustado                398.8  2.6      391.5  3.9  13251.1  1227.1   
Depósitos ajustados            399.4  2.6      391.7  3.7  13250.5  1208.8   
Visa pagado suavizado          397.2  3.9      388.2  6.4  14102.0  1485.5   

                      shuffle_split                       ensamble_5  
                               mean   std veces_mejor ganancia_max_M  
version                                                               
Actual                        198.9  11.8           7          400.5  
Sin fechas                    200.1  11.7          41          403.3  
Sueldo ajustado               198.6  11.8           0          399.9  
Depósitos ajustados           198.9  11.8           0          400.5  
Visa pagado suavizado         197.8  11.7           2          397.3

In [6]:
# --- 6. Anotar ---
anotar(tabla, "z011 drift", [202104])

Anotado en resultados_experimentos.csv


## Conclusiones (08/10)
Diferencia contra Actual (10 semillas):
- Sin fechas: **+1,6 M**, gana 6/10 semillas, Shuffle Split 41/50 → leve mejora, dentro del ruido.
- Sueldo ajustado (×2/3 en 202106): −0,6 M, 0/10 → no cambia (el árbol ya lo manejaba).
- Depósitos ajustados: +0,0 M → no cambia.
- Visa pagado suavizado: −2,2 M, 3/10 → empeora.
- Conclusión: el drift de estas variables casi no afecta. Sacar las fechas de cierre es lo único con algo de señal positiva.